In [14]:
import os
import faiss
import joblib
import numpy as np

from sentence_transformers import SentenceTransformer
from openai import OpenAI

In [15]:
# Emotion Detection
tfidf = joblib.load("../models/tfidf_vectorizer.pkl")
emotion_model = joblib.load("../models/main_emotion_model_tfidf.pkl")
main_encoder = joblib.load("../models/main_emotion_encoder.pkl")

# RAG
index = faiss.read_index("../models/counseling_faiss.index")
counsel_df = joblib.load("../models/counseling_dataset.pkl")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L12-v2"
)

print("All models loaded successfully.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

All models loaded successfully.


In [17]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [18]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

In [20]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": "Hello"
        }
    ]
)

print(response.choices[0].message.content)

Hello. How can I help you today?


In [27]:
def get_ai_response(user_text):

    # ---------------- Emotion Prediction ----------------
    x = tfidf.transform([user_text])

    pred = emotion_model.predict(x)[0]

    main_emotion = main_encoder.inverse_transform([pred])[0]

    # ---------------- RAG ----------------
    query_embedding = embedding_model.encode(
        [user_text],
        convert_to_numpy=True
    )

    distances, indices = index.search(query_embedding, 3)
    contexts = []

    for idx in indices[0]:
        contexts.append(counsel_df.iloc[idx]["Response"])

    rag_context = "\n\n".join(contexts)

    # ---------------- Prompt ----------------
    prompt = f"""
You are a personalized AI Study Assistant.

Detected Emotion:
{main_emotion}

Relevant Counseling Context:
{rag_context}

User Message:
{user_text}

Instructions:
- Generate a supportive, personalized response.
- Do NOT copy the retrieved text.
- Use it only as background knowledge.
- Use ONLY the information provided in the user's message and the retrieved context.
- Do NOT invent personal details or past events.
- If information is unavailable, do not assume it.
"""

    # ---------------- Groq ----------------
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful AI Study Assistant."
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return {
    "main_emotion": main_emotion,
    "rag_context": contexts,
    "reply": response.choices[0].message.content
}

In [28]:
result = get_ai_response("I failed my exam and I feel hopeless.")

print(result["main_emotion"])
print(result["reply"])

Sad
I'm so sorry to hear that you failed your exam and are feeling hopeless. It's completely understandable to feel that way, but I want you to know that this one setback doesn't define your worth or abilities. It's a temporary bump in the road, and you can always try again.

Remember that failing is a natural part of the learning process, and many people have been in your shoes before. It doesn't mean you're not capable or smart; it just means you might need to approach things differently next time.

Take a deep breath and try not to be too hard on yourself. Instead, focus on what you can learn from this experience and how you can use it to grow. You got this, and you can come out of this even stronger and more resilient. Would you like to talk more about what's been going on and how you're feeling? I'm here to listen and support you.
